# Flystabilitet og turbulensrespons

## Trim, phugoid og longitudinelle egenmoder

### Pilotprosjekt for Matematikk 1 og flyingeniør

Et lett skolefly flyr i trimmet, horisontal flukt. Først skal vi bestemme hvordan vinge, haleplan og motor må balansere krefter og momenter. Deretter undersøker vi hvordan flyet reagerer når det utsettes for et høyderorsutslag eller et vertikalt vindkast.

Prosjektet har fem deler:

1. **Trim og lastfordeling:** et lineært system for vinge, haleplan og motorkraft
2. **Phugoid:** en dempet skalar oscillator for langsom fart- og høydebevegelse
3. **Longitudinell flydynamikk:** en firetilstands vektor-ODE
4. **Egenmoder:** phugoid og kortperiodisk bevegelse
5. **Stabilitet og design:** tyngdepunkt, halevirkning, vindkast og enkel tilbakekobling

### Læringsmål

Etter prosjektet skal du kunne

- formulere kraft- og momentbalanse som $Ax=b$,
- kontrollere fortegn, enheter og residualer,
- beregne trimkrefter og trimkoeffisienter,
- skrive en andreordens ODE som et førsteordenssystem,
- bruke Euler på en skalar og en vektoriell flymodell,
- tolke en tilstandsrommodell $\dot x=Ax+Bu$,
- beregne egenverdier og egenvektorer,
- skille mellom phugoid og kortperiodisk mode,
- beregne periode, naturlig frekvens og dempingsforhold,
- simulere høyderorspuls og vertikalt vindkast,
- undersøke hvordan tyngdepunkt og tilbakekobling påvirker stabiliteten.

### Viktig avgrensning

Dette er en linearisert undervisningsmodell omkring én trimtilstand. Parameterne er avrundede og valgt for tydelige, stabile flymoder. Modellen skal ikke brukes til design, sertifisering, flygeplanlegging eller vurdering av et virkelig luftfartøys flygeegenskaper.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

rho = 1.112       # kg/m^3, representativ høyde
g = 9.81          # m/s^2
m = 1100.0        # kg
W = m*g           # N
S = 16.2          # m^2, vingeareal
c_bar = 1.50      # m, middelkorde
U0 = 55.0         # m/s, trimhastighet
q_dyn = 0.5*rho*U0**2

# Referanseflyet

Vi bruker et lett én-motors skolefly med følgende idealiserte data:

- masse: $1100$ kg
- trimhastighet: $55$ m/s
- vingeareal: $16.2$ m²
- horisontalhaleareal: $3.8$ m²
- middelkorde: $1.50$ m
- vingens løftangrepspunkt: $0.15$ m bak tyngdepunktet
- haleplanets angrepspunkt: $4.2$ m bak tyngdepunktet
- motorens kraftlinje: $0.25$ m under tyngdepunktet
- motstand i trim: $750$ N

Positive løftkrefter virker oppover. Positiv motorkraft virker fremover. Vi velger positivt pitchmoment som nese opp.

In [ ]:
S_h = 3.8
x_w = 0.15       # m bak CG
x_h = 4.20       # m bak CG
z_T = -0.25      # m, motorlinje under CG
D_trim = 750.0   # N

# Del A: Trim og lastfordeling

## A.1 Kraft- og momentbalanse

De ukjente er

$$
x_{trim}=
\begin{pmatrix}
L_v\\L_h\\T
\end{pmatrix},
$$

med vingeløft $L_v$, haleplanskraft $L_h$ og motorkraft $T$.

Trimkravene er:

$$L_v+L_h=W,$$

$$-x_wL_v-x_hL_h-z_TT=0,$$

$$T=D_{trim}.$$

Haleplanskraften kan bli negativ, som betyr at halen virker nedover.

## Oppgave A1: Bygg og løs trimsystemet

Skriv systemet som

$$A_{trim}x_{trim}=b_{trim}.$$

Løs og kontroller residualet.

In [ ]:
A_trim = np.array([
    [..., ..., 0.0],
    [..., ..., ...],
    [0.0, 0.0, ...]
], dtype=float)

b_trim = np.array([W, 0.0, D_trim])
x_trim = ...

L_v, L_h, T_trim = x_trim
print("Vingeløft:", L_v, "N")
print("Haleplanskraft:", L_h, "N")
print("Motorkraft:", T_trim, "N")
print("Residualnorm:", ...)

## A.2 Løftkoeffisienter

Løftkoeffisienten defineres som

$$
\boxed{C_L=\frac{L}{qS}.}
$$

Beregn vingens og haleplanets trimkoeffisienter.

In [ ]:
CL_v = ...
CL_h = ...

print("Vingens CL:", CL_v)
print("Haleplanets CL:", CL_h)

## Oppgave A2: Tyngdepunkt og fysisk trim

Flytt tyngdepunktet ved å endre armene til vinge og hale:

- fremre CG: $x_w=0.30$ m, $x_h=4.35$ m
- referanse-CG
- bakre CG: $x_w=-0.05$ m, $x_h=4.00$ m

For hvert tilfelle, beregn:

- haleplanskraft
- vingeløft
- haleplanets $C_L$
- om trimløsningen virker fysisk rimelig

Diskuter hvorfor en matematisk løsning ikke nødvendigvis er en tillatt trimtilstand.

In [ ]:
def løs_trim(xw, xh, zT=z_T, drag=D_trim):
    A = np.array([
        [1.0, 1.0, 0.0],
        [-xw, -xh, -zT],
        [0.0, 0.0, 1.0]
    ])
    return np.linalg.solve(A, np.array([W, 0.0, drag]))

# Beregn de tre CG-tilfellene.

## A.3 Lastplassering som lineær algebra

Flyet har fire lastplasser med masser $m_i$ og posisjoner $x_i$. Tyngdepunktet er

$$
\boxed{x_G=\frac{x^Tm}{\mathbf1^Tm}.}
$$

Bruk plasseringene

$$x=(-1.2,-0.2,0.8,2.4)\ \mathrm m$$

for pilot, passasjer, bagasje og drivstoff. Undersøk hvordan drivstofforbruk og bagasje påvirker tyngdepunktet.

In [ ]:
x_last = np.array([-1.2, -0.2, 0.8, 2.4])
m_last = np.array([85.0, 75.0, 40.0, 120.0])

x_G_last = ...
print("Lastsystemets tyngdepunkt:", x_G_last, "m")

# Valgfri del A4: Løftfordeling over vingen

Vingen deles i seksjoner. Den ukjente sirkulasjonen er

$$
\Gamma=(\Gamma_1,\ldots,\Gamma_n)^T.
$$

En gitt påvirkningsmatrise beskriver hvordan alle vingeseksjoner påvirker hverandre:

$$
\boxed{A_\Gamma\Gamma=b_\alpha.}
$$

Denne delen kan brukes som en flyfaglig fordypning. Studentene sammenligner rektangulær, avsmalnende og geometrisk vridd vinge, og beregner seksjonsløft fra

$$L_i'=\rho U_0\Gamma_i.$$

# Del B: Phugoid som skalar ODE

Phugoid er en langsom utveksling mellom kinetisk og potensiell energi. Vi bruker høydeavviket $h(t)$:

$$
\boxed{
\ddot h+2\zeta_p\omega_p\dot h+\omega_p^2h=0.}
$$

En enkel frekvenstilnærming er

$$
\boxed{\omega_p\approx\frac{\sqrt2g}{U_0}.}
$$

Vi bruker dempingsforholdet

$$\zeta_p=0.055.$$

In [ ]:
zeta_p = 0.055
omega_p = np.sqrt(2)*g/U0
T_p_omtrent = 2*np.pi/omega_p

print("Phugoid vinkelfrekvens:", omega_p, "rad/s")
print("Udempet periode:", T_p_omtrent, "s")

## B.1 Førsteordensform

Sett

$$x_1=h,\qquad x_2=\dot h.$$

Da får vi

$$
\boxed{
\dot x=A_px,\qquad
A_p=
\begin{pmatrix}
0&1\\
-\omega_p^2&-2\zeta_p\omega_p
\end{pmatrix}.}
$$

## Oppgave B1: Egenverdier for hånd og i Python

Beregn egenverdiene, periodetiden og halveringstiden til amplituden. Sammenlign med

$$
\lambda=-\zeta_p\omega_p\pm i\omega_p\sqrt{1-\zeta_p^2}.
$$

In [ ]:
A_p = np.array([
    [0.0, 1.0],
    [..., ...]
])

lambda_p, P_p = ...
print("Egenverdier:", lambda_p)

# Beregn dempet periode og amplitudens halveringstid.

## B.2 Euler-simulering

Start med et høydeavvik på 20 m og null vertikalhastighet:

$$x(0)=(20,0)^T.$$

Simuler 600 sekunder.

In [ ]:
def euler_system(f, x0, sluttid, dt):
    n = int(round(sluttid/dt))
    t = np.linspace(0.0, n*dt, n+1)
    X = np.zeros((n+1, len(x0)))
    X[0] = x0
    for k in range(n):
        X[k+1] = ...
    return t, X


def phugoid_ode(t, x):
    return A_p@x

# Simuler med flere tidssteg og plott høydeavviket.

## Oppgave B2: Trimfart og flymode

Gjenta for trimhastighetene 40, 55 og 75 m/s. Hvordan endres phugoidperioden? Forklar forholdet mellom høyere fart og langsommere energiutveksling.

# Del C: Longitudinell vektor-ODE

Vi bruker små avvik fra trim:

$$
\boxed{x=(u,w,q,\theta)^T,}
$$

med

- $u$: avvik i fremoverhastighet, m/s
- $w$: vertikalt hastighetsavvik i kroppsfaste akser, m/s
- $q$: pitchrate, rad/s
- $\theta$: pitchvinkelavvik, rad

Tilstandsmodellen er

$$
\boxed{\dot x=Ax+B_e\delta_e+B_gw_g.}
$$

$\delta_e$ er høyderorsutslag og $w_g$ er vertikalt vindkast.

## C.1 Undervisningsmatrise

Vi bruker den avrundede, stabile matrisen

$$
A=
\begin{pmatrix}
-0.030&0.080&0&-9.81\\
-0.120&-0.550&54.0&0\\
0.006&-0.095&-1.15&0\\
0&0&1&0
\end{pmatrix}.
$$

Den er valgt slik at modellen får ett langsomt phugoidpar og ett raskere kortperiodisk par.

In [ ]:
A_lon = np.array([
    [-0.030,  0.080,  0.0, -9.81],
    [-0.120, -0.550, 54.0,  0.0],
    [ 0.006, -0.095, -1.15,  0.0],
    [ 0.0,    0.0,    1.0,   0.0]
])

B_e = np.array([0.0, -6.0, -2.8, 0.0])
B_g = np.array([0.0, 0.55, 0.015, 0.0])

## Oppgave C1: Les hver matriserad

Forklar hvordan radene representerer:

1. fremoverhastighet og tyngdekraftkobling
2. vertikal hastighet og pitchrate
3. pitchmoment
4. den kinematiske sammenhengen $\dot\theta=q$

Finn matrisens determinant, rang og egenverdier.

In [ ]:
print("det(A) =", ...)
print("rang(A) =", ...)
print("egenverdier =", ...)

## C.2 Styre- og vindprofiler

Vi bruker en kort høyderorspuls og et separat vertikalt vindkast.

Fortegnene er pedagogiske. Et negativt høyderorsutslag gir i denne modellen en nese-opp-respons.

In [ ]:
def høyderor_puls(t):
    return np.deg2rad(-2.0) if 2.0 <= t < 3.0 else 0.0


def vindkast(t):
    return 3.0 if 80.0 <= t < 83.0 else 0.0


def longitudinell_ode(t, x):
    return A_lon@x+B_e*høyderor_puls(t)+B_g*vindkast(t)

## Oppgave C2: Simuler flyresponsen

Start i trim, $x(0)=0$, og simuler 300 sekunder. Plott alle fire tilstander. Bruk grader for $q$ og $\theta$ i figurene.

Identifiser:

- rask respons rett etter høyderorspulsen
- langsom respons som blir igjen
- responsen på vindkastet

In [ ]:
# Bruk euler_system til å simulere longitudinell_ode.
# Lag fire delplott med norske aksetekster.

## C.3 Fra kroppshastighet til angrepsvinkel

For små avvik kan angrepsvinkelavviket tilnærmes med

$$
\boxed{\Delta\alpha\approx\frac{w}{U_0}.}
$$

Beregn og plott $\Delta\alpha$ i grader. Sammenlign med pitchvinkelen.

In [ ]:
# alpha_deg = np.rad2deg(X[:,1]/U0)

# Del D: Egenmoder

For en egenverdi

$$\lambda=\sigma+i\omega$$

bruker vi

$$
\boxed{\omega_n=\sqrt{\sigma^2+\omega^2},}
$$

$$
\boxed{\zeta=-\frac{\sigma}{\omega_n},}
$$

$$
\boxed{T=\frac{2\pi}{|\omega|}.}
$$

De to komplekse parene forventes å representere phugoid og kortperiodisk mode.

## Oppgave D1: Klassifiser modene

Beregn for hvert komplekst par:

- naturlig frekvens
- dempingsforhold
- periode
- amplitudens halveringstid

Klassifiser det langsomme paret som phugoid og det raske som kortperiodisk mode.

In [ ]:
egenverdier_lon, P_lon = np.linalg.eig(A_lon)

# Sorter komplekse egenverdier etter absolutt imaginærdel.
# Beregn modaldata uten å telle konjugerte par dobbelt.

## D.2 Tolk egenvektorene

Egenvektorer har komponenter med forskjellige enheter. Skaler derfor tilstandene med

$$
S_x=\operatorname{diag}(U_0,U_0,0.2,0.2).
$$

Undersøk de skalerte egenvektorene og vurder hvilke tilstander som dominerer:

- phugoid: særlig fart og pitchvinkel
- kortperiode: særlig $w$, angrepsvinkel og pitchrate

In [ ]:
S_x = np.diag([U0, U0, 0.2, 0.2])
P_skalert = np.linalg.solve(S_x, P_lon)

# Normaliser hver søyle etter største absoluttverdi og tolk.

## Oppgave D2: Direkte og modal simulering

Bruk en initialtilstand som inneholder både en fartforstyrrelse og en pitchrate. Sammenlign:

- direkte Euler på $\dot x=Ax$
- modal simulering med $A=PDP^{-1}$
- en simulering der bare phugoidmoden beholdes
- en simulering der bare kortperiodemoden beholdes

In [ ]:
# Kontroller først at egenvektormatrisen er inverterbar.
# Implementer modal simulering med komplekse koordinater og ta realdelen til slutt.

# Del E: Stabilitet og design

## E.1 Tyngdepunkt som parameter

Et bakre tyngdepunkt reduserer ofte flyets statiske pitchstabilitet. I undervisningsmodellen representerer vi dette ved å gjøre $M_w$ mindre negativt:

$$
M_w(\chi)=-0.095+0.070\chi,
$$

der

- $\chi=0$: referanse-CG
- $\chi=1$: betydelig bakre CG
- $\chi<0$: fremre CG

In [ ]:
def A_med_CG(chi):
    A = A_lon.copy()
    A[2,1] = -0.095+0.070*chi
    A[2,2] = -1.15+0.20*chi
    return A

## Oppgave E1: Stabilitetskart

La $\chi$ variere fra $-0.5$ til $1.8$. For hver verdi:

- beregn egenverdiene
- finn største realdel
- plott største realdel mot $\chi$
- finn omtrent hvor en mode blir ustabil

Sammenlign med trimresultatene fra Del A. Diskuter forskjellen mellom å kunne trimme flyet og å ha tilfredsstillende dynamisk stabilitet.

In [ ]:
chi_grid = np.linspace(-0.5, 1.8, 200)
maks_realdel = ...

# Plott og marker stabilitetsgrensen Re(lambda)=0.

## E.2 Enkel pitchdemper

Bruk tilbakekoblingen

$$
\boxed{\delta_e=-k_q q-k_\theta\theta.}
$$

Med

$$K=(0,0,k_q,k_\theta)$$

blir lukket systemmatrise

$$
\boxed{A_{cl}=A-B_eK.}
$$

In [ ]:
k_q = 0.12
k_theta = 0.08
K_feedback = np.array([[0.0, 0.0, k_q, k_theta]])
A_lukket = ...

print("Åpen sløyfe:", np.linalg.eigvals(A_lon))
print("Lukket sløyfe:", np.linalg.eigvals(A_lukket))

## Oppgave E2: Velg en pitchdemper

Sammenlign flere verdier av $k_q$ og $k_\theta$. Vurder:

- demping av kortperiodemoden
- påvirkning på phugoid
- maksimal høyderorskommando
- oversving
- om for sterk tilbakekobling gir en uønsket respons

Dette er en parameterstudie, ikke et generelt regulator­design.

# Fordypning 1: Løftelinje og vingevridning

Bygg en enkel påvirkningsmatrise for $n$ vingeseksjoner og løs

$$A_\Gamma\Gamma=b_\alpha.$$

Sammenlign:

- rektangulær vinge
- avsmalnende vinge
- negativ geometrisk vridning mot vingespissen
- konsekvensene for løftfordeling og indusert last nær vingespissen

# Fordypning 2: Lateral stabilitet

En lateral modell kan bruke

$$x=(\beta,p,r,\phi)^T$$

og gi tre karakteristiske bevegelser:

- rollmode
- Dutch roll
- spiral mode

Dette kan bli en egen utvidelse når studentene har arbeidet med longitudinell dynamikk.

# Fordypning 3: Fladder

En vingeseksjon med bøyning og torsjon kan skrives

$$
M\ddot q+C(V)\dot q+K(V)q=0.
$$

Som førsteordenssystem blir systemmatrisen avhengig av flyhastigheten. Fladderhastigheten kan idealisert identifiseres når en egenverdis realdel krysser null.

Denne delen krever mer aerodynamikk og strukturdynamikk og passer best i et senere emne.

# Modellkritikk

Diskuter minst åtte punkter:

- Trimmodellen samler vinge- og halekrefter i punktkrefter.
- Motstand og motorkraft er sterkt forenklet.
- Aerodynamiske koeffisienter antas lineære nær trim.
- Atmosfærens tetthet holdes konstant.
- Flyet behandles som et stivt legeme.
- Longitudinell og lateral bevegelse er frakoblet.
- Tilstandsmodellen gjelder bare for små avvik.
- Stabilitetsderivertene er undervisningsparametre.
- Vindkastet er romlig uniformt og stykkevis konstant.
- Høyderorsaktuatoren har ingen dynamikk eller metning.
- Bakkeeffekt, stall og kompressibilitet er utelatt.
- Phugoidtilnærmingen beskriver ikke kortperiodisk bevegelse.
- Euler krever kontroll av tidssteget.
- Egenvektorer med ulike enheter må skaleres før fysisk tolkning.
- Tilbakekoblingen er ikke et sertifisert autopilotsystem.

## Mulige videreføringer

- aktuator som ekstra ODE
- thrust-dynamikk
- høyde og flygebanevinkel
- ikke-lineære løft- og motstandsfunksjoner
- stallmodell
- målte stabilitetsderiverte
- systemidentifikasjon fra flytestdata
- lateral dynamikk
- fladder
- flyging gjennom tilfeldig turbulens

# Oppsummering

Skriv en kort rapport der du forklarer

1. hvordan kraft- og momentbalansen ga et lineært trimsystem,
2. hvordan tyngdepunktet påvirket haleplanskraften,
3. hvordan phugoid ble en dempet oscillator,
4. hvordan den longitudinelle modellen ble skrevet som $\dot x=Ax+Bu$,
5. hvordan egenverdiene skilte phugoid fra kortperiodisk mode,
6. hvorfor egenvektorene måtte skaleres,
7. hvordan høyderorspuls og vindkast eksiterte ulike modi,
8. hvordan et bakre tyngdepunkt påvirket stabiliteten,
9. hvordan pitchtilbakekobling flyttet egenverdiene,
10. hvorfor en trimbar flytilstand ikke nødvendigvis har gode dynamiske egenskaper.

## Faglig bakgrunn

Prosjektet følger standardideen om å linearisere longitudinelle flyligninger omkring en trimmet flytilstand. Slike modeller gir normalt en langsom phugoidmode og en raskere kortperiodisk mode. Studentmodellen bruker en avrundet tilstandsmatrise for å gjøre lineær algebra, ODE-er og modal tolkning synlig.